# controls

> Native form controls styled through their own states, raised to press and sunken to type into


In [ ]:
#| default_exp controls

Every form control we need is already a native HTML tag, and the browser already keeps track of its state. We style those tags directly. For their states we use the browser's pseudo-classes, such as `:disabled` and `:user-invalid`, and ARIA attributes, such as `aria-busy` and `aria-pressed`, rather than tracking state ourselves. Variants are attributes with no value, such as `secondary`.

In [ ]:
from fastcore.utils import *
from fasthtml.common import Div, A, Label, Input, Small, Form, Textarea, Select, Option, Legend, Menu, Li, Code, unqid
from fastlucide.icons import Check, Copy as CopyIcon
from answerui.registry import reg, hypelt
from answerui.tokens import relief
from answerui.demo import example
from answerui.layout import Stack, Cluster
import answerui.base  # the reset this styling is written on

In [ ]:
from fasthtml.common import *
from answerui.demo import demo
from answerui.registry import reg
from fastcore.test import *
from fastlucide.icons import X

## Button

We give buttons depth: a raised face shows that a button can be pressed, and a sunken face shows that it is already pressed. A press moves only the button's label, bevels and shadows, never its overall size, so a row of buttons stays level however many of them are down.

Some presses last longer than a click. A button stays down while it has any of these attributes: `aria-busy`, set while a request is in flight; `aria-pressed`, set while a toggle is on; and `aria-expanded`, set while a popover or menu it opened is open. Only the press itself moves the label: a button held down by one of these attributes stays sunken with its label centred.

A busy button also shows a spinner, without growing or changing its wording to make room for it. The label fades, the spinner takes its place, and the label returns when the request ends. A busy button can still be clicked: `aria-busy` changes only how it looks. Set `disabled` on a control that must not be activated. A button that makes its own htmx request becomes busy without `aria-busy`, because htmx adds the `htmx-request` class to it while the request runs. That busy face appears only after `--busy-delay`, 200ms by default, so a request that finishes sooner shows nothing. A button with a server-set `aria-busy` looks busy at once. Every held state and the delayed busy face use the `--face-held` and `--shadow-held` that a variant sets. When a form makes the request, htmx adds `htmx-request` to the form, not to its button. To make the button busy, set the form's `hx-indicator` to the button.

A disabled button is flat and faded, with no shadow, because there is nothing to press. It keeps its colour, so a disabled primary button still looks like the primary. When it is enabled, it is raised again.

The default button is the primary action and takes the accent colour. A `secondary` button has the neutral surface colour and a hairline border instead. Use `quiet` for actions that should not compete with the content. A quiet button has no face, border, or shadow at rest and inherits its parent's text colour. It shows a tint of that colour when hovered, and twice that tint when pressed or latched.

Give a link the `button` attribute to style it as a button, variants included. It cannot be disabled or busy, since a link has no such states.

In [ ]:
_button_css = r"""
& {
  --face: var(--v); --face-held: var(--v2); --shadow-held: var(--bevel-in), var(--sunken); --_shift: 0px; --busy-delay: 200ms;
  --pad-block: var(--space-2); --pad-inline: var(--space);
  display: inline-flex; align-items: center; justify-content: center; gap: var(--space-2);
  padding: calc(var(--pad-block) + var(--_shift)) calc(var(--pad-inline) - var(--_shift))
           calc(var(--pad-block) - var(--_shift)) calc(var(--pad-inline) + var(--_shift));
  font-weight: var(--weight1); line-height: var(--leading-1);
  background: var(--face); color: var(--n-3); text-decoration: none;
  border: 1px solid transparent; border-radius: var(--radius);
  box-shadow: var(--bevel), var(--raised);
  transition: box-shadow calc(var(--motion) / 2) var(--ease), padding calc(var(--motion) / 2) var(--ease), background var(--motion) var(--ease), opacity var(--motion) var(--ease), -webkit-text-fill-color var(--motion) var(--ease);
}
&:hover { --face: var(--v1); }
&:is(:active, [aria-busy=true], [aria-pressed=true], [aria-expanded=true]) { --face: var(--face-held); box-shadow: var(--shadow-held); }
&:active { --_shift: calc(1px * var(--relief)); transition-duration: calc(var(--motion) / 4); }
&:disabled { opacity: .5; --_shift: 0px; box-shadow: none; }
&.htmx-request:not([aria-busy=true]) { animation: aui-sink 0s linear var(--busy-delay) both; }
"""

Button = hypelt('Button', sel='button, a[button]', layer='components', doc='Raised at rest, down while active, busy, or expanded, latched by `aria-pressed`; `secondary` for the lesser action; a link carrying `button` is styled the same way.', css=_button_css)

reg.rule(r'''
button {
  &:is([aria-busy=true], .htmx-request) { position: relative; }
  &[aria-busy=true] { -webkit-text-fill-color: transparent; }
  > svg { transition: opacity var(--motion) var(--ease); }
  &:is([aria-busy=true], .htmx-request) { > svg { opacity: 0; }
    &::before { content: ''; position: absolute; inset: 0; margin: auto; inline-size: 1.25em; block-size: 1.25em;
      border: 2px solid currentColor; border-inline-end-color: transparent; border-radius: var(--radius-full);
      animation: aui-spin calc(var(--motion) * 5) linear infinite; } }
  &.htmx-request:not([aria-busy=true]) { > svg { transition-delay: var(--busy-delay); }
    &::before { animation: aui-spin calc(var(--motion) * 5) linear infinite, aui-busy-in var(--motion) var(--ease) var(--busy-delay) both; } }
}
@keyframes aui-sink { to { --face: var(--face-held); box-shadow: var(--shadow-held); -webkit-text-fill-color: transparent; } }
@keyframes aui-busy-in { from { opacity: 0; } }
''', layer='components')

d9adcfc7082f [components]

In [ ]:
Button.variant('secondary', r"""
--face: var(--n-2); color: var(--n2); border-color: var(--n-1);
--face-held: var(--n-2);
&:hover { --face: var(--v-3); }
&:is([aria-pressed=true], [aria-expanded=true]) { --face: var(--v-3); color: var(--v2); }
""", layer='components', doc='The lesser action: the neutral surface with a hairline border, and the accent wash when latched.')

Button.secondary [components] :is(button, a[button])[secondary]: The lesser action: the neutral surface with a hairline border, and the accent wash when latched.

In [ ]:
Button.variant('quiet', r'''
--face: transparent; color: inherit; border-color: transparent; box-shadow: none;
--face-held: oklch(from var(--tint) l c h / calc(alpha * 2)); --shadow-held: none;
&:hover { --face: var(--tint); }
&:is(:active, [aria-busy=true], [aria-pressed=true], [aria-expanded=true]) { --face: var(--face-held); }
&:active { --_shift: 0px; }
''', layer='components', doc='No face, border, or shadow at rest, for an action that should not compete: dismiss, close, and toolbar buttons.')

Button.quiet [components] :is(button, a[button])[quiet]: No face, border, or shadow at rest, for an action that should not compete: dismiss, close, and toolbar buttons.

Each variant and state in a wrapping cluster: default, secondary, quiet, a link, danger and success contexts, disabled, busy, and latched:


In [ ]:
@example(Button)
def _button_example():
    return Cluster(Button('Save'), Button('Cancel', secondary=True), Button('Dismiss', quiet=True), A('Sign up', href='#', button=True), Button('Delete', danger=True), Button('Publish', success=True),
        Button('Locked', disabled=True), Button('Save', aria_busy='true'), Button('Pinned', aria_pressed='true'), Button('Pinned', secondary=True, aria_pressed='true'), Button('Pinned', quiet=True, aria_pressed='true'),
        gap='var(--space-2)')


In [ ]:
Button.demo()

```html
<div cluster style="--gap: var(--space-2);">
<button>Save</button><button secondary>Cancel</button><button quiet>Dismiss</button><a href="#" button>Sign up</a><button danger>Delete</button><button success>Publish</button><button disabled>Locked</button><button aria-busy="true">Save</button><button aria-pressed="true">Pinned</button><button secondary aria-pressed="true">Pinned</button><button quiet aria-pressed="true">Pinned</button></div>

```

Application styling can change the resting padding without rewriting the pressed state. Here `compact` tightens the padding and `flat` sets `--relief` to 0. Because they are declared here, they belong to this example, not to the built-in button. `doc(Button)` lists them for as long as they are registered. To reduce depth across the whole application instead, set `:root { --relief: .5; }`.

In [ ]:
Button.variant('compact', '--pad-block: var(--space-3); --pad-inline: var(--space-2);', doc='Tighter padding.')
Button.variant('flat', '--relief: 0;', doc='No rims, shadows, or displacement.')

Button.flat [app] :is(button, a[button])[flat]: No rims, shadows, or displacement.

The flat buttons keep their colour states without rims, shadows, or displacement; the two pin buttons show unselected and selected states:

In [ ]:
demo(Cluster(
    Button('Compact', compact=True), Button('Flat', flat=True),
    Button('Pin', secondary=True, flat=True, aria_pressed='false'),
    Button('Pin', secondary=True, flat=True, aria_pressed='true'),
    gap='var(--space-2)'))

```html
<div cluster style="--gap: var(--space-2);">
<button compact>Compact</button><button flat>Flat</button><button secondary flat aria-pressed="false">Pin</button><button secondary flat aria-pressed="true">Pin</button></div>

```

A button that holds nothing but an icon should be square, with its inline padding equal to its block padding. Give it an `aria-label`, because the icon gives it no name. The rule depends on that label: a selector can't see text, so `svg:only-child` alone would also match a button with an icon and a text label:

In [ ]:
Button.rule('icon-only', '&[aria-label]:has(> svg:only-child) { --pad-inline: var(--pad-block); }', layer='components', doc='A button with an aria-label whose only child element is an svg is square; a button with an icon and a text label keeps the usual padding.')

Button.icon-only [components] button, a[button]: A button with an aria-label whose only child element is an svg is square; a button with an icon and a text label keeps the usual padding.

The three close buttons have an `aria-label` and only an icon, so they are square. The button with an icon and the text Close keeps the usual padding, as the text-only Save button does:

In [ ]:
close = [Button(X(), aria_label='Close'), Button(X(), aria_label='Close', secondary=True), Button(X(), aria_label='Close', quiet=True)]
demo(Cluster(*close, Button(X(), 'Close', secondary=True), Button('Save'), gap='var(--space-2)'))

```html
<div cluster style="--gap: var(--space-2);">
<button aria-label="Close"><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg></button><button aria-label="Close" secondary><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg></button><button aria-label="Close" quiet><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg></button><button secondary><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg>Close</button><button>Save</button></div>

```

## Text controls

Where a button is raised, a text control is sunken. Inverting a button's lighting makes the control look like a well cut into the page: somewhere to put something, not something to press. A disabled well is drawn flat, as a disabled button is. These rules apply to the tags themselves, so a form written by hand looks the same as one built with `Field`. Text-like inputs, textareas and selects are wells; an `input` used as a button, and a file picker, keep their native look.

A number input is a well without spin buttons, because a number is never entered with them. The keyboard's arrows still step the value.

Invalid controls use the danger context. The browser matches `:user-invalid` once the user has interacted with an invalid control, and the server can set `aria-invalid="true"`. Either one switches the control to the danger hue, so its border takes that colour with no rule naming it:

In [ ]:
reg.context(':user-invalid, [aria-invalid=true]', 'var(--hue-danger)')

_text_sel = 'input:is(:not([type]), [type=text], [type=email], [type=password], [type=search], [type=url], [type=tel], [type=number], [type=date], [type=time], [type=datetime-local], [type=month], [type=week])'
reg.rule(r"""
:is(%(text_sel)s, textarea, select, [well]) {
  inline-size: 100%%; padding: var(--space-2) var(--space-1);
  --face: var(--n-3); background: var(--face); color: var(--n2);
  border: 1px solid var(--n-1); border-radius: var(--radius); box-shadow: var(--bevel-in), var(--sunken);
  transition: border-color var(--motion) var(--ease), box-shadow var(--motion) var(--ease);
}
:is(input, textarea, select):is(:user-invalid, [aria-invalid=true]) { border-color: var(--v); }
:is(%(text_sel)s, textarea, select):disabled { --face: var(--n-2); color: var(--n); box-shadow: var(--bevel-in); }
::placeholder { color: var(--n1); }
input[type=number] { appearance: textfield; &::-webkit-inner-spin-button, &::-webkit-outer-spin-button { appearance: none; margin: 0; } }
""" %dict(text_sel=_text_sel), name='text-controls', layer='components', doc='The well shared by text inputs, textareas, selects, and any element marked `well`, taking the danger hue when invalid.')

Input = hypelt('Input', sel=_text_sel, layer='components', doc='A text-like input, styled as a well by the text-controls rule.')
Textarea = hypelt('Textarea', layer='components', css='min-block-size: 5rem; resize: vertical; field-sizing: content;',
    doc='A textarea, styled as a well by the text-controls rule; it grows with its content.')
Select = hypelt('Select', layer='components', doc='A select, styled as a well by the text-controls rule.')

Each text control on its own, at rest, and disabled:

In [ ]:
@example(Input)
def _input_example(): return Stack(Input(placeholder='Search', type='search', aria_label='Search'), Input(value='Locked', disabled=True, aria_label='Locked'), style='inline-size:16rem')

@example(Textarea)
def _textarea_example(): return Textarea('A note that grows with its content.', aria_label='Note', style='inline-size:16rem')

@example(Select)
def _select_example(): return Select(Option('Free'), Option('Team'), aria_label='Plan', style='inline-size:16rem')

In [ ]:
Input.demo()

```html
<div stack style="inline-size:16rem">
  <input placeholder="Search" type="search" aria-label="Search">
  <input value="Locked" disabled aria-label="Locked">
</div>

```

To draw an element that isn't a control as a well, such as a third party's widget mounted into a plain `div` or a read-only block of text, give it the `well` attribute:

In [ ]:
demo(Stack(Div('A widget mounts here', well=True), Input(placeholder='A text input, for comparison', aria_label='Comparison')))

```html
<div stack>
  <div well>A widget mounts here</div>
  <input placeholder="A text input, for comparison" aria-label="Comparison">
</div>

```

## Enabling a button

A Save button that is live before there is anything to save invites a click that does nothing. So render the submit button `disabled` and set `enable` to say when it becomes enabled: `changed` once any control in its form differs from the value it was rendered with, `valid` once the form passes its own validation, or both. The `enable` rule compares each control's value with the default that the browser keeps from the rendered page, so it keeps no copy of the rendered values. Resetting the form disables the button again. A form that htmx swaps in works the same way: its rendered values are its defaults.


In [ ]:
_enable_js = r"""
const f = el.form ?? el;
const run = () => {
    const changed = [...f.elements].some(c => c.matches('input:not([type=hidden], [type=submit], [type=button], [type=reset]), select, textarea')
        && (c.type === 'checkbox' || c.type === 'radio' ? c.checked !== c.defaultChecked
            : c.tagName === 'SELECT' ? [...c.options].some(o => o.selected !== o.defaultSelected) : c.value !== c.defaultValue));
    const valid = [...f.elements].every(c => c.validity.valid);  // not checkValidity(), which fires `invalid` on each control and the validity rule would show its message
    for (const b of f.querySelectorAll('button[enable]')) {
        const wants = b.getAttribute('enable').split(' ');
        b.disabled = !wants.every(w => w === 'changed' ? changed : w === 'valid' ? valid : true);
    }
};
e.type === 'reset' ? setTimeout(run) : run();"""

reg.rule(name='enable', on={('form:has(button[enable])', 'input'): _enable_js, ('form:has(button[enable])', 'change'): _enable_js, ('form:has(button[enable])', 'reset'): _enable_js, ('button[enable]', 'init'): _enable_js},
    layer='components', doc="A submit button rendered disabled with `enable='changed'`, `enable='valid'`, or both comes alive when its form has changed, or is valid, and goes back when it is not.")

enable [components]: A submit button rendered disabled with `enable='changed'`, `enable='valid'`, or both comes alive when its form has changed, or is valid, and goes back when it is not.

To see it work, change the name or type an email address. Save is enabled only once the form has changed and is valid; the email is required. Reset restores the rendered values and disables Save again:


In [ ]:
demo(Form(Input(name='name', value='Ada', aria_label='Display name'), Input(name='email', type='email', required=True, placeholder='Email', aria_label='Email'),
    Cluster(Button('Save', type='submit', disabled=True, enable='changed valid'), Button('Reset', type='reset', secondary=True)), stack=True, style='inline-size: 24rem'))


```html
<form enctype="multipart/form-data" stack style="inline-size: 24rem">  <input name="name" value="Ada" aria-label="Display name">
  <input name="email" type="email" required placeholder="Email" aria-label="Email">
  <div cluster>
<button type="submit" disabled enable="changed valid">Save</button><button type="reset" secondary>Reset</button>  </div>
</form>
```

## Group

Controls that act as one share their edges: only the outer corners are rounded, and neighbours meet at a single hairline. `Toolbar` puts a row of commands, such as a pager or a set of toggles, in an HTML `menu`, with each command in an item. The server sets which toggles are down with `aria-pressed`; `Toolbar` only lays them out. `Group` joins a text input and its one action, such as a search box and its button, or a name and a Create button. The input takes the width the button leaves. While the input has focus, the focus ring surrounds the whole group, because the pair acts as one control.

Use `inset` to put the button inside the input's well instead, for a small action on the text, such as copying or clearing it. In either form, render the button `disabled` with `enable`, so that at rest the group looks like one field rather than a pair of controls.


In [ ]:
_group_css = r"""
display: inline-flex; align-items: stretch; margin: 0; padding: 0; list-style: none; border-radius: var(--radius); --_radius: var(--radius); --_small: var(--radius-1);
> li { display: flex; }
> *, > li > * { --radius: 0; }
> :first-child, > li:first-child > * { border-start-start-radius: var(--_radius); border-end-start-radius: var(--_radius); }
> :last-child, > li:last-child > * { border-start-end-radius: var(--_radius); border-end-end-radius: var(--_radius); }
> * + * { margin-inline-start: -1px; }
&:has(> :is(input, select)) { inline-size: 100%; }
> :is(input, select) { flex: 1; min-inline-size: 0; }
> :is(button, a[button]) { flex: none; }
&:has(> :is(input, select):focus-visible) { outline: var(--ring); outline-offset: 2px; }
> :is(input, select):focus-visible { outline: none; }
"""

@hypelt(sel='[group]', layer='components', css=_group_css)
def Group(*c, **attrs):
    "A text input and its action sharing an edge; `inset` puts the action inside the input's well"
    return Div(*c, group=True, **attrs)

@hypelt(sel='menu[group]', layer='components')
def Toolbar(*controls, **attrs):
    "A row of commands sharing their edges: a `menu` with each control as an item"
    return Menu(*[Li(c) for c in controls], group=True, **attrs)

Group.variant('inset', r"""
--face: var(--n-3); background: var(--face); border: 1px solid var(--n-1); box-shadow: var(--bevel-in), var(--sunken);
align-items: center; padding-inline-end: var(--space-3);
& > * { margin: 0; --radius: var(--_small); }
& > :first-child, & > :last-child { border-radius: var(--_small); }
& > :is(input, select) { border: 0; background: transparent; box-shadow: none; }
& > :is(button, a[button]), & > [tooltip] > :is(button, a[button]) { --pad-block: var(--space-3); --pad-inline: var(--space-2); }
""", layer='components', doc='The action inside the well; the action starts disabled and `enable` wakes it.')

Group.inset [components] :is([group])[inset]: The action inside the well; the action starts disabled and `enable` wakes it.

Two groups: a name with its Create button, and a name being edited in an inset well. Type a name into the first to enable Create. Change the name in the second to enable its Save:


In [ ]:
@example(Group)
def _group_example():
    return Stack(Form(Group(Input(name='name', placeholder='Instance name', required=True, aria_label='Instance name'), Button('Create', type='submit', disabled=True, enable='valid'))),
        Form(Group(Input(name='name', value='personal', aria_label='Name'), Button('Save', type='submit', disabled=True, enable='changed valid'), inset=True)),
        style='inline-size: 24rem')

In [ ]:
Group.demo()

```html
<div stack style="inline-size: 24rem">
<form enctype="multipart/form-data">    <div group>
      <input name="name" placeholder="Instance name" required aria-label="Instance name">
<button type="submit" disabled enable="valid">Create</button>    </div>
</form><form enctype="multipart/form-data">    <div group inset>
      <input name="name" value="personal" aria-label="Name">
<button type="submit" disabled enable="changed valid">Save</button>    </div>
</form></div>

```

Two toolbars: a pager, and a toggle bar with one button latched and one disabled. The toggle bar is named with `aria-label`, because its buttons are toggles rather than one form choice. For an exclusive choice, use native radios. The server normally sets a toggle's state; here a demo-only rule flips `aria-pressed` on click instead:


In [ ]:
reg.rule(name='demo-toolbar-toggle', sel='menu[group]', on={('& button[aria-pressed]', 'click'): r"el.setAttribute('aria-pressed', String(el.getAttribute('aria-pressed') !== 'true'))"},
    doc='Demo only: flips aria-pressed in place of the server.')

demo-toolbar-toggle [app] menu[group]: Demo only: flips aria-pressed in place of the server.

In [ ]:
@example(Toolbar)
def _toolbar_example():
    return Cluster(Toolbar(Button('Previous', secondary=True, type='button'), Button('Next', secondary=True, type='button'), aria_label='Pages'),
        Toolbar(Button('Bold', secondary=True, aria_pressed='true'), Button('Italic', secondary=True, aria_pressed='false'),
            Button('Underline', secondary=True, aria_pressed='false', disabled=True), aria_label='Text style'))

In [ ]:
Toolbar.demo()

```html
<div cluster>
<menu group aria-label="Pages">    <li>
<button secondary type="button">Previous</button>    </li>
    <li>
<button secondary type="button">Next</button>    </li>
</menu><menu group aria-label="Text style">    <li>
<button secondary aria-pressed="true">Bold</button>    </li>
    <li>
<button secondary aria-pressed="false">Italic</button>    </li>
    <li>
<button secondary aria-pressed="false" disabled>Underline</button>    </li>
</menu></div>

```

## Copy

`Copy` is a quiet button, icon-only by default, that copies a value meant to be pasted elsewhere, such as an API key, a share link or a command. Set `copyfor` to the id of the element to copy: `Copy` copies a control's value, or any other element's text. Or set `copy` to fixed text to copy. For 1.5s after a click, the icon is a check and the button has the `copied` attribute. A button with an `aria-label`, such as an icon-only one, is also named 'Copied' for that time, and screen readers announce the new name. A button with a text label keeps its text as its name. On `http:` a click does nothing, because the Clipboard API needs a secure origin.

In [ ]:
_copy_js = r"""
const t = document.getElementById(el.getAttribute('copyfor'));
const text = el.hasAttribute('copy') ? el.getAttribute('copy') : t?.matches('input, textarea, select') ? t.value : t?.textContent;
if (text == null) return;
await navigator.clipboard.writeText(text);
el._label ??= el.getAttribute('aria-label') ?? '';
el.toggleAttribute('copied', true); if (el._label) el.setAttribute('aria-label', 'Copied');
clearTimeout(el._copied);
el._copied = setTimeout(() => { el.removeAttribute('copied'); if (el._label) el.setAttribute('aria-label', el._label) }, 1500);"""

_copy_css = r"""
> svg > use:last-child { display: none; }
&[copied] > svg > use:first-child { display: none; }
&[copied] > svg > use:last-child { display: inline; }
"""

@hypelt(sel='button:is([copyfor], [copy])', layer='components', css=_copy_css, on={('&', 'click'): _copy_js})
def Copy(
    *c,           # Label text; with none, the button is square and named Copy
    copyfor=None, # id of the element whose value, or text, is copied
    copy=None,    # Fixed text to copy instead of an element's
    **attrs,      # Attributes for the button; `secondary` replaces the default `quiet`
):
    "A quiet button that copies fixed text, or an element's value or text, to the clipboard"
    if 'secondary' not in attrs: attrs.setdefault('quiet', True)
    if not c: attrs.setdefault('aria_label', 'Copy')
    return Button(CopyIcon(aria_hidden='true') + Check(), *c, type='button', copyfor=copyfor, copy=copy, **attrs)

The first example is an API key in a read-only well, with the copy button inset. The second is a command, with a labelled `secondary` button beside it. Click either button to see the check:

In [ ]:
@example(Copy)
def _copy_example():
    key, cmd = unqid(), unqid()
    return Stack(Group(Input(value='sk-live-4f9c2e7b1a0d', readonly=True, id=key, aria_label='API key'), Copy(copyfor=key), inset=True),
        Cluster(Code('pip install answerui', id=cmd), Copy('Copy', copyfor=cmd, secondary=True), gap='var(--space-2)'),
        style='inline-size: 24rem')

In [ ]:
Copy.demo()

```html
<div stack style="inline-size: 24rem">
  <div group inset>
    <input value="sk-live-4f9c2e7b1a0d" readonly aria-label="API key" id="_8ttACRlvStmWMDoxTzDnMA" name="_8ttACRlvStmWMDoxTzDnMA">
<button type="button" copyfor="_8ttACRlvStmWMDoxTzDnMA" quiet aria-label="Copy"><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" aria-hidden="true" class="lucide-icon "><use href="#lc-copy"></use><use href="#lc-check"></use></svg></button>  </div>
  <div cluster style="--gap: var(--space-2);">
<code id="_IOOHLcJCQuWfB66KlgxqKQ">pip install answerui</code><button type="button" copyfor="_IOOHLcJCQuWfB66KlgxqKQ" secondary><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" aria-hidden="true" class="lucide-icon "><use href="#lc-copy"></use><use href="#lc-check"></use></svg>Copy</button>  </div>
</div>

```

## Click to edit

A value that is read far more often than it is changed should not sit in a well, which invites typing, nor in plain text, which gives no sign it can change. An `editable` button shows the value in a box with the same size, padding and corners as the well that replaces it, but with a dashed hairline in place of the sunken face. No field exists until the click, and nothing depends on hover. When the user clicks, the app fetches the form and swaps it in place of the value: a single well for a name, or a whole form for a value made of parts. Inside a form, give the `editable` button `type="button"`. Name the action with `aria-label`, because the button's text is the value:


In [ ]:
Button.variant('editable', r'''
--face: transparent; --pad-block: var(--space-2); --pad-inline: var(--space-1);
--face-held: oklch(from var(--tint) l c h / calc(alpha * 2)); --shadow-held: none;
inline-size: 100%; justify-content: start; text-align: start;
color: inherit; font: inherit; cursor: text;
border: 1px dashed var(--n); box-shadow: none;
&:hover { --face: var(--tint); }
&:is(:active, [aria-busy=true], [aria-pressed=true], [aria-expanded=true]) { --face: var(--face-held); }
&:active { --_shift: 0px; }
''', layer='components', doc='A value drawn as the dashed ghost of the well it becomes; click to edit, and the app swaps the form in.')

Button.editable [components] :is(button, a[button])[editable]: A value drawn as the dashed ghost of the well it becomes; click to edit, and the app swaps the form in.

To demonstrate this without a server, a demo-only rule swaps each value for its form and back. The first value is a name: clicking it replaces it with a well and a Save button that is enabled once the text changes. The second is a schedule: clicking it opens a small form with a day and a time. Either way, saving shows the new value in place of the form, and clicking anywhere else or pressing Escape restores the old one:


In [ ]:
reg.rule(name='demo-editable', sel='[editable-demo]', on={
    ('& button[editable]', 'click'): r"const p = el.closest('[editable-demo]'); p.firstElementChild.hidden = true; p.lastElementChild.hidden = false; p.querySelector('input, select').focus()",
    ('& form', 'submit'): r"e.preventDefault(); const p = el.closest('[editable-demo]'); p.querySelector('button[editable]').textContent = el.dataset.text.replace(/\{(\w+)\}/g, (_, n) => el.elements[n].value); el.hidden = true; p.firstElementChild.hidden = false",
    ('& form', 'keydown'): r"if (e.key === 'Escape') { el.reset(); el.hidden = true; el.previousElementSibling.hidden = false }",
    (':root', 'pointerdown'): r"for (const f of document.querySelectorAll('[editable-demo] > form:not([hidden])')) if (!f.contains(e.target)) { f.reset(); f.hidden = true; f.previousElementSibling.hidden = false }",
}, doc='Demo only: swaps the value for its form and back in place of the server; a click outside or Escape cancels.')

def _editable_demo(value, form, text, **attrs):
    "An editable value and the hidden form that edits it; `text` is the template that turns the form's fields into the new value"
    return Div(Button(value, editable=True, type='button', **attrs), Form(form, data_text=text, hidden=True), editable_demo=True)


In [ ]:
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
demo(Stack(
    _editable_demo('Untitled-1', Group(Input(name='name', value='Untitled-1', aria_label='Name'), Button('Save', type='submit', disabled=True, enable='changed valid'), inset=True),
        '{name}', aria_label='Rename Untitled-1'),
    _editable_demo('Every Monday at 09:00', Group(Select(*[Option(d, selected=d == 'Monday') for d in days], name='day', aria_label='Day'),
        Input(type='time', name='time', value='09:00', aria_label='Time'), Button('Save', type='submit', disabled=True, enable='changed valid')),
        'Every {day} at {time}', aria_label='Change the schedule'),
    style='inline-size: 24rem'))


```html
<div stack style="inline-size: 24rem">
  <div editable-demo>
<button editable type="button" aria-label="Rename Untitled-1">Untitled-1</button><form enctype="multipart/form-data" data-text="{name}" hidden>      <div group inset>
        <input name="name" value="Untitled-1" aria-label="Name">
<button type="submit" disabled enable="changed valid">Save</button>      </div>
</form>  </div>
  <div editable-demo>
<button editable type="button" aria-label="Change the schedule">Every Monday at 09:00</button><form enctype="multipart/form-data" data-text="Every {day} at {time}" hidden>      <div group>
<select name="day" aria-label="Day"><option selected>Monday</option><option>Tuesday</option><option>Wednesday</option><option>Thursday</option><option>Friday</option></select>        <input type="time" name="time" value="09:00" aria-label="Time">
<button type="submit" disabled enable="changed valid">Save</button>      </div>
</form>  </div>
</div>

```

## Switch

A switch usually turns something on or off straight away, where a checkbox records a choice for a form to submit. The two are the same control to the browser: a switch is a checkbox with `role="switch"`, and the browser supplies the state, the keyboard handling, and the announcement. `Switch` only sets `type="checkbox"` and `role="switch"`. The track is a well and the thumb is raised, as a button is. Turning it on colours the track and slides the thumb across. In a `Field` it sits before its label, as a checkbox does:

In [ ]:
_switch_css = r"""
appearance: none; --face: var(--n-3); inline-size: 2.75em; block-size: 1.5em; margin: 0; padding: 2px;
background: var(--face); border: 1px solid var(--n-1); border-radius: var(--radius-full); box-shadow: var(--bevel-in), var(--sunken);
display: inline-grid; align-items: center;
transition: background var(--motion) var(--ease), box-shadow var(--motion) var(--ease);
&::before { content: ''; --face: var(--n-2); inline-size: calc(1.5em - 6px); block-size: calc(1.5em - 6px); border-radius: var(--radius-full);
  background: var(--face); box-shadow: var(--bevel), var(--raised);
  transition: translate var(--motion) var(--ease), background var(--motion) var(--ease); }
&:checked { --face: var(--v); }
&:checked::before { translate: 1.25em 0; --face: var(--n-3); }
&:disabled { --face: var(--n-2); box-shadow: var(--bevel-in); }
&:disabled::before { --face: var(--n-1); box-shadow: var(--bevel); }
"""

@hypelt(sel='input[role=switch]', layer='components', css=_switch_css)
def Switch(**attrs):
    "A checkbox with the switch role"
    return Input(type='checkbox', role='switch', **attrs)

Off, on, and disabled:

In [ ]:
@example(Switch)
def _switch_example():
    return Cluster(Switch(aria_label='Off'), Switch(checked=True, aria_label='On'), Switch(disabled=True, aria_label='Disabled'))

In [ ]:
Switch.demo()

```html
<div cluster>
  <input type="checkbox" role="switch" aria-label="Off">
  <input type="checkbox" role="switch" checked aria-label="On">
  <input type="checkbox" role="switch" disabled aria-label="Disabled">
</div>

```

## Checkbox and radio

The browser's own checkbox and radio follow the hue, but they have none of our depth, so we set `appearance: none` and restyle them ourselves. Each is a small well. At this size we drop the rims and keep only the sunken shadow. `Checkbox` and `Radio` only set the type:

In [ ]:
_choice_css = r"""
input[type=checkbox]:not([role=switch]), input[type=radio] { appearance: none; --face: var(--n-3); inline-size: 1.15em; block-size: 1.15em; margin: 0;
  background: var(--face); border: 1px solid var(--n-1); border-radius: var(--radius-1); box-shadow: var(--sunken);
  display: inline-grid; place-content: center;
  transition: background var(--motion) var(--ease), border-color var(--motion) var(--ease); }
input[type=checkbox]:not([role=switch])::before, input[type=radio]::before { content: ''; inline-size: .6em; block-size: .6em; background: var(--n-3); scale: 0; transition: scale var(--motion) var(--ease); }
:is(input[type=checkbox]:not([role=switch]), input[type=radio]):disabled { --face: var(--n-2); border-color: var(--n-1); }
"""
reg.rule(_choice_css, name='choice-controls', layer='components', doc='The small well and the mark shared by checkboxes and radios.')

_checkbox_css = r"""
&::before { clip-path: polygon(14% 44%, 0 65%, 50% 100%, 100% 16%, 80% 0%, 43% 62%); }
&:is(:checked, :indeterminate) { --face: var(--v); border-color: var(--v); }
&:is(:checked, :indeterminate)::before { scale: 1; }
&:indeterminate::before { clip-path: none; block-size: .15em; }
&:is(:checked, :indeterminate):disabled { --face: var(--n-1); border-color: var(--n-1); }
"""

@hypelt(sel='input[type=checkbox]:not([role=switch])', layer='components', css=_checkbox_css)
def Checkbox(**attrs):
    "A checkbox, styled as a small well by the `choice-controls` rule"
    return Input(type='checkbox', **attrs)


In [ ]:
_radio_css = r"""
border-radius: var(--radius-full);
&::before { border-radius: var(--radius-full); background: var(--v); }
&:checked { border-color: var(--v); }
&:checked::before { scale: 1; }
&:checked:disabled::before { background: var(--n); }
"""

@hypelt(sel='input[type=radio]', layer='components', css=_radio_css)
def Radio(**attrs):
    "A radio button, styled as a small well by the `choice-controls` rule"
    return Input(type='radio', **attrs)

Unchecked, checked, disabled, and checked and disabled:

In [ ]:
@example(Checkbox)
def _checkbox_example():
    return Cluster(Checkbox(aria_label='Unchecked'), Checkbox(checked=True, aria_label='Checked'),
        Checkbox(disabled=True, aria_label='Disabled'), Checkbox(checked=True, disabled=True, aria_label='Checked and disabled'))


In [ ]:
Checkbox.demo()

```html
<div cluster>
  <input type="checkbox" aria-label="Unchecked">
  <input type="checkbox" checked aria-label="Checked">
  <input type="checkbox" disabled aria-label="Disabled">
  <input type="checkbox" checked disabled aria-label="Checked and disabled">
</div>

```

Radios in a group, unchecked, checked, and checked and disabled:

In [ ]:
@example(Radio)
def _radio_example():
    return Cluster(Radio(name='r', aria_label='Unchecked'), Radio(name='r', checked=True, aria_label='Checked'), Radio(name='rd', checked=True, disabled=True, aria_label='Checked and disabled'))

In [ ]:
Radio.demo()

```html
<div cluster>
  <input type="radio" name="r" aria-label="Unchecked">
  <input type="radio" name="r" checked aria-label="Checked">
  <input type="radio" name="rd" checked disabled aria-label="Checked and disabled">
</div>

```

## Range

A slider is the browser's `input[type=range]`, restyled as a well with a raised thumb. The one thing we add is a fill along the track up to the thumb. Because CSS can't read where the thumb is, a small behavior computes the fill from the slider's `value`, `min` and `max`, and writes it to `--p` on `init`, on `input`, and when the form is reset. A script that sets `value` must then dispatch an `input` event to update the fill, because setting `value` doesn't fire that event.

The browser exposes the track and the thumb as pseudo-elements inside the input, not as elements of their own. WebKit, which Chrome and Safari share, names them `::-webkit-slider-runnable-track` and `::-webkit-slider-thumb`, and Firefox names them `::-moz-range-track` and `::-moz-range-thumb`. The two spellings need separate rules, because a browser drops a whole rule whose selector list names a pseudo-element it does not know. `_range_parts` generates the rules for one engine's names, and `_range_css` calls it once for each engine. `relief` gives each part rims derived from its own face colour: the page colour for the track and the accent for the thumb. `_range_css` handles two engine differences on its own: it moves WebKit's thumb down to the centre of the track, where Firefox already puts it, and it removes the border that Firefox gives the thumb.

In [ ]:
_fill = r"const lo = el.min === '' ? 0 : +el.min, hi = el.max === '' ? 100 : +el.max; el.style.setProperty('--p', (hi > lo ? (el.value - lo) / (hi - lo) * 100 : 0) + '%')"
_range_on = {
    ('&', 'init'): _fill,
    ('&', 'input'): _fill,
    ('form:has(input[type=range])', 'reset'): r"""
await new Promise(r => setTimeout(r, 0));
for (const range of [...el.elements].filter(i => i.matches('input[type=range]'))) { const el = range; """ + _fill + " }"}

def _range_parts(track, thumb):
    "The track and thumb rules for one engine's pseudo-elements"
    return relief('&::%s, &::%s' % (track, thumb)) + r"""
&::%(track)s { --face: var(--n-3); block-size: var(--track); border-radius: var(--radius-full); border: 1px solid var(--n-1);
  background: linear-gradient(to right, var(--v) var(--p), var(--face) 0); box-shadow: var(--bevel-in), var(--sunken); }
&::%(thumb)s { appearance: none; --face: var(--v); inline-size: var(--thumb); block-size: var(--thumb); border-radius: var(--radius-full);
  background: var(--face); box-shadow: var(--bevel), var(--raised); transition: box-shadow calc(var(--motion) / 2) var(--ease); }
&:active::%(thumb)s { box-shadow: var(--bevel-in), var(--sunken); }
&:disabled::%(track)s { background: linear-gradient(to right, var(--n) var(--p), var(--n-2) 0); }
&:disabled::%(thumb)s { --face: var(--n-1); box-shadow: var(--bevel); }
""" %dict(track=track, thumb=thumb)

_range_css = r"""
appearance: none; inline-size: 100%; block-size: 1.5em; margin: 0; background: none;
--track: .6em; --thumb: 1.25em; --p: 0%;
&::-webkit-slider-thumb { margin-block-start: calc((var(--track) - var(--thumb)) / 2 - 1px); }
&::-moz-range-thumb { border: 0; }
""" + _range_parts('-webkit-slider-runnable-track', '-webkit-slider-thumb') + _range_parts('-moz-range-track', '-moz-range-thumb')

@hypelt(sel='input[type=range]', layer='components', on=_range_on, css=_range_css)
def Range(**attrs):
    "A slider"
    return Input(type='range', **attrs)

A slider at 35, a disabled one at 65, one at 8 of 10, and one at -5 in a range that ends at zero. The last shows that `max=0` is treated as a bound, not as a missing one.

In [ ]:
@example(Range)
def _range_example():
    return Stack(Range(value=35, aria_label='Brightness'), Range(value=65, disabled=True, aria_label='Locked'), Range(min=0, max=10, value=8, aria_label='Rating'),
        Range(min=-10, max=0, value=-5, aria_label='Offset'),
        style='inline-size:16rem')

In [ ]:
Range.demo()

```html
<div stack style="inline-size:16rem">
  <input type="range" value="35" aria-label="Brightness">
  <input type="range" value="65" disabled aria-label="Locked">
  <input type="range" min="0" max="10" value="8" aria-label="Rating">
  <input type="range" min="-10" max="0" value="-5" aria-label="Offset">
</div>

```

## Field

A control needs a label, often a line of help, and sometimes an error. `Field` connects each of these to the control so that assistive technology reads them together. It wraps the control in its label, which associates the two without an id, and puts a checkbox, radio or switch before its label text rather than after. For a checkbox, radio or switch, the label is only as wide as the control and its text, so a click in the empty space beside it does nothing. A `hint` and an `error` appear beneath the control, and their ids are added to its `aria-describedby`. `Field` shows an `error` in the danger colour and sets `aria-invalid="true"` on the control. What counts as invalid is the application's decision; `Field` only shows the message it is given.


In [ ]:
_field_css = r"""
&, > label { display: grid; gap: var(--space-3); font-size: var(--text-1); }
> label { font-weight: var(--weight1); }
> small { font-weight: var(--weight); color: var(--n1); }
> label:has(> :is([type=checkbox], [type=radio])) { grid-template-columns: auto 1fr; justify-self: start; align-items: center; gap: var(--space-2); font-weight: var(--weight); }
&:has(:disabled) > label { color: var(--n); }
"""

@hypelt(sel='[field]', layer='components', css=_field_css)
def Field(
    label,      # Label text
    control,    # The control
    hint=None,  # Help text shown under the control
    error=None, # Error message; marks the control invalid
    **attrs,    # Attributes for the field
):
    "A labelled control with optional help and error text"
    boxed = control.attrs.get('type') in ('checkbox', 'radio')
    descriptions = []
    if hint: descriptions.append(Small(hint, id=True))
    if error:
        descriptions.append(Small(error, id=True, danger=True))
        control(aria_invalid='true')
    if descriptions:
        ids = [control.attrs.get('aria-describedby', ''), *(o.id for o in descriptions)]
        control(aria_describedby=' '.join(filter(None, ids)))
    return Div(Label(*([control, label] if boxed else [label, control])), *descriptions, field=True, **attrs)

reg.rule('small[danger] { color: var(--v2); }', layer='components')


e398d9b03997 [components]

The browser shows a bubble for an invalid control only when the user tries to submit. With `enable='valid'`, Save stays disabled while the form is invalid, so the bubble never appears. `Field`'s `validity` rule shows the browser's validation message under the field in place of the bubble, once the user leaves the control invalid or tries to submit. On a submit attempt, every invalid field shows its message and the first takes focus, as it would with the bubble. While the message is shown, the control's `aria-describedby` lists the message's id, as it does for an error that `Field` renders. When the value is valid again, the rule removes the message, and its id from the control's `aria-describedby`. The rule leaves an `error` passed to `Field` alone, because that message has no `validation` attribute.

In [ ]:
_validity_js = r"""
const f = el.closest('[field]'); let m = f.querySelector(':scope > small[validation]');
const ids = (el.getAttribute('aria-describedby') || '').split(' ').filter(i => i && i !== m?.id);
const show = e.type === 'invalid' || (e.type === 'focusout' ? el.matches(':user-invalid') : m && !el.validity.valid);
if (!show) { m?.remove(); ids.length ? el.setAttribute('aria-describedby', ids.join(' ')) : el.removeAttribute('aria-describedby'); return }
if (!m) { m = document.createElement('small'); m.toggleAttribute('validation', true); m.toggleAttribute('danger', true); m.id = 'validation-' + Math.random().toString(36).slice(2, 8); f.append(m) }
m.textContent = el.validationMessage; el.setAttribute('aria-describedby', [...ids, m.id].join(' '));
if (e.type !== 'invalid') return;
e.preventDefault(); if ([...el.form?.elements ?? [el]].find(c => !c.validity.valid) === el) el.focus();"""
_ctl = '& :is(input, select, textarea)'
Field.rule('validity', on={(_ctl, 'focusout'): _validity_js, (_ctl, 'input'): _validity_js, (_ctl, 'invalid', True): _validity_js}, layer='components',
    doc="When the user leaves a control invalid or tries to submit, the browser's own validation message appears under the field in place of its bubble, listed in the control's `aria-describedby`; both go when the value is valid again.")

Field.validity [components] [field]: When the user leaves a control invalid or tries to submit, the browser's own validation message appears under the field in place of its bubble, listed in the control's `aria-describedby`; both go when the value is valid again.

When a server returns the form with a message about an invalid submission, render the message as a `small` with the `validation` attribute, outside any field. The `form-validation` rule removes it as soon as the user edits a control, because it described the values that were sent. A `validation` message inside a field belongs to that field's control: the `validity` rule removes it once the control is valid.

In [ ]:
reg.rule(name='form-validation', sel='form', on={('& :is(input, select, textarea)', 'input'): r"for (const m of el.form.querySelectorAll('small[validation]')) if (!m.closest('[field]')) m.remove()"},
    layer='components', doc='A validation message a form came back with, a `small` marked `validation` outside any field, clears once the user edits a control, since it described the values that were sent.')

form-validation [components] form: A validation message a form came back with, a `small` marked `validation` outside any field, clears once the user edits a control, since it described the values that were sent.

Every kind of control in one form. The username keeps its hint alongside a server-supplied error. Type an invalid address into the email field and leave the field to see the browser's validation message:

In [ ]:
@example(Field)
def _field_example():
    return Form(
        Field('Name', Input(name='name', placeholder='Ada Lovelace')),
        Field('Email', Input(name='email', type='email', required=True), hint='We never share it.'),
        Field('Notes', Textarea(name='notes', placeholder='Anything else?')),
        Field('Plan', Select(Option('Free'), Option('Team'), name='plan')),
        Field('Send me updates', Checkbox(name='updates', checked=True)),
        Field('Email me about mentions', Switch(name='mentions', checked=True)),
        Field('Weekly digest', Radio(name='digest', checked=True)),
        Field('Volume', Range(name='volume')),
        Field('Region', Input(name='region', value='Locked', disabled=True)),
        Field('Username', Input(name='username', value='taken'), hint='Shown on your profile.', error='That name is in use.'),
        Cluster(Button('Save'), Button('Cancel', secondary=True, type='button'), gap='var(--space-2)'),
        stack=True)


In [ ]:
Field.demo()

```html
<form enctype="multipart/form-data" stack>  <div field>
<label>Name      <input name="name" placeholder="Ada Lovelace">
</label>  </div>
  <div field>
<label>Email      <input name="email" type="email" required aria-describedby="_NKodL-ldSTq4kEvB7CG3Uw">
</label><small id="_NKodL-ldSTq4kEvB7CG3Uw">We never share it.</small>  </div>
  <div field>
<label>Notes<textarea name="notes" placeholder="Anything else?"></textarea></label>  </div>
  <div field>
<label>Plan<select name="plan"><option>Free</option><option>Team</option></select></label>  </div>
  <div field>
<label>      <input type="checkbox" name="updates" checked>
Send me updates</label>  </div>
  <div field>
<label>      <input type="checkbox" role="switch" name="mentions" checked>
Email me about mentions</label>  </div>
  <div field>
<label>      <input type="radio" name="digest" checked>
Weekly digest</label>  </div>
  <div field>
<label>Volume      <input type="range" name="volume">
</label>  </div>
  <div field>
<label>Region      <input name="region" value="Locked" disabled>
</label>  </div>
  <div field>
<label>Username      <input name="username" value="taken" aria-invalid="true" aria-describedby="_QAkHaqjmQ4OVueaKm1q50g _XhNNi9xlS3Cx2UNDiBi9AA">
</label><small id="_QAkHaqjmQ4OVueaKm1q50g">Shown on your profile.</small><small danger id="_XhNNi9xlS3Cx2UNDiBi9AA">That name is in use.</small>  </div>
  <div cluster style="--gap: var(--space-2);">
<button>Save</button><button secondary type="button">Cancel</button>  </div>
</form>
```

A variant declared on `Field` applies to the field's outer `div`, which is where `Field` puts the attributes passed to it. The control keeps its own attributes, and the ids already in its `aria-describedby` stay, before those of the hint and error:

In [ ]:
notes = Textarea(name='notes', aria_describedby='notes-policy')
field = Field('Notes', notes, hint='Keep it brief.', error='Please remove the account number.', compact=True)
assert notes.attrs['aria-invalid'] == 'true'
assert notes.attrs['aria-describedby'].split() == ['notes-policy', *(o.id for o in field.children[1:])]
test_eq(field.attrs['compact'], True)
notes.attrs


{'name': 'notes',
 'aria-describedby': 'notes-policy _3Ct0udhdR0C_0SeB6Ob-Eg _aPm3OoaYQc2ocN-MPtAhzA',
 'aria-invalid': 'true'}

## Fieldset

A fieldset groups fields under a legend. It has a hairline border and its own padding. The layout sets the spacing between its fields, as everywhere else:

In [ ]:
Fieldset = hypelt('Fieldset', layer='components', doc='A bordered group of fields under a legend.', css=r'''
margin: 0; min-inline-size: 0; border: 1px solid var(--n-1); border-radius: var(--radius); padding: var(--space);
''')
reg.rule('legend { font-weight: var(--weight1); padding-inline: var(--space-2); }', layer='components')

67ad4fd35e5f [components]

Two checkbox fields under a legend:

In [ ]:
@example(Fieldset)
def _fieldset_example():
    return Fieldset(Legend('Notifications'),
        Field('Email me about mentions', Input(type='checkbox', checked=True)),
        Field('Email me a weekly digest', Input(type='checkbox')),
        stack=True, style='--gap: var(--space-2)')

In [ ]:
Fieldset.demo()

```html
<fieldset stack style="--gap: var(--space-2)"><legend>Notifications</legend>  <div field>
<label>      <input type="checkbox" checked>
Email me about mentions</label>  </div>
  <div field>
<label>      <input type="checkbox">
Email me a weekly digest</label>  </div>
</fieldset>
```

A field sometimes matters only while a checkbox is checked, such as a limit. Set the checkbox's `reveals` to that field's id, as `commandfor` names a dialog by id: the field is shown while the checkbox is checked and hidden while it is not. Render the field `hidden` or not to match the checkbox's initial state; the `reveals` rule keeps the two in step on each change:

In [ ]:
reg.rule(name='reveals', on={('input[reveals]', 'change'): r"document.getElementById(el.getAttribute('reveals')).hidden = !el.checked"}, layer='components',
    doc="A checkbox or switch with `reveals='<id>'` shows that element while checked and hides it while not; render the element `hidden` to match the initial state.")

reveals [components]: A checkbox or switch with `reveals='<id>'` shows that element while checked and hides it while not; render the element `hidden` to match the initial state.

In [ ]:
demo(Form(Field('Set a monthly limit', Checkbox(name='limit_on', reveals='limit')),
    Field('Monthly limit (US$)', Input(name='limit', type='number', value=200), id='limit', hidden=True), stack=True))

```html
<form enctype="multipart/form-data" stack>  <div field>
<label>      <input type="checkbox" name="limit_on" reveals="limit">
Set a monthly limit</label>  </div>
  <div field hidden id="limit">
<label>Monthly limit (US$)      <input name="limit" type="number" value="200">
</label>  </div>
</form>
```

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()